# Phishing URL Detector with Machine Learning

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Load dataset
url = "https://raw.githubusercontent.com/chamanthmvs/Phishing-Website-Detection/master/extracted_csv_files/phishing-urls.csv"
df = pd.read_csv(url)

# Clean column names
df.columns = df.columns.str.strip()

# Use 'Domain' column as the URL input, and 'label' as the target
print("Using columns:", df.columns.tolist())

# Feature extraction function
def extract_features(domain):
    return {
        'domain_length': len(domain),
        'num_dots': domain.count('.'),
        'has_ip': int(bool(re.search(r'\d+\.\d+\.\d+\.\d+', domain))),
        'has_at': int('@' in domain),
        'has_hyphen': int('-' in domain),
        'has_https': int('https' in domain or 'http' in domain)
    }

# Extract features and prepare labels
features = df['Domain'].astype(str).apply(lambda x: pd.Series(extract_features(x)))
X = features
y = df['label'].map({1: 1, 0: 0})  # 1: phishing, 0: legitimate

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
print("\nModel Evaluation:")
print(classification_report(y_test, y_pred))

# Test on new domains
test_domains = [
    "login-secure-paypal.com",
    "google.com",
    "198.51.100.1",
    "amazon.com-login-update.info"
]
test_df = pd.DataFrame([extract_features(domain) for domain in test_domains])
preds = model.predict(test_df)

# Display predictions
print("\nTest Domain Results:")
for domain, label in zip(test_domains, preds):
    print(f"{domain} --> {'Phishing' if label else 'Safe'}")
